# Activity 3: ResNet-18 with a Poincaré BMLR Head

In this activity, we train a hybrid image classifier on MNIST:

$$
\text{image}\xrightarrow{\text{ResNet-18}}z\in\mathbb{R}^{512}
\xrightarrow{\operatorname{CLIP}+\operatorname{Exp}_0^K}x\in\mathbb{P}^{512}_K
\xrightarrow{\text{BMLR}}\text{class logits}.
$$

The goal is to isolate the geometric classification head. We use a small MNIST subset so that the experiment finishes quickly on a Colab GPU.


## Mathematical recap

For class $k$, BMLR uses the logit introduced in Activity 2,

$$
u_k(x)=-\alpha_k B^{v_k}(x)+b_k,
\qquad
p(y=k\mid x)=\frac{\exp(u_k(x))}{\sum_j\exp(u_j(x))}.
$$

On the Poincaré ball with curvature $K<0$,

$$
B^v(x)=\frac{1}{\sqrt{-K}}\log\left(
\frac{\lVert v-\sqrt{-K}x\rVert^2}
{1+K\lVert x\rVert^2}
\right).
$$

ResNet produces a Euclidean feature $z$. Before the exponential map, we clip its norm:

$$
c(z;r)=\min\left\{1,\frac{r}{\lVert z\rVert}\right\},
\qquad
\operatorname{CLIP}(z;r)=c(z;r)z,
\qquad
x=\operatorname{Exp}_0^K\!\left(\operatorname{CLIP}(z;r)\right).
$$

Here $r>0$ is the **clipping radius**, while $c(z;r)$ is the sample-wise **clip factor**. Clipping limits how far the Euclidean feature travels under the exponential map and mitigates vanishing gradients in a hybrid Euclidean–hyperbolic classifier. We use $K=-1$ and $r=1$, matching the Poincaré image-classification setting in the HBNN implementation.


## References

- Ziheng Chen, Bernhard Schölkopf, and Nicu Sebe. **Hyperbolic Busemann Neural Networks.** CVPR 2026. [Paper](https://arxiv.org/abs/2602.18858) · [Code](https://github.com/GitZH-Chen/HBNN)
- Yunhui Guo, Xudong Wang, Yubei Chen, and Stella X. Yu. **Clipped Hyperbolic Classifiers Are Super-Hyperbolic Classifiers.** CVPR 2022. [Paper](https://openaccess.thecvf.com/content/CVPR2022/html/Guo_Clipped_Hyperbolic_Classifiers_Are_Super-Hyperbolic_Classifiers_CVPR_2022_paper.html)


## Setup

The notebook uses PyTorch and TorchVision from the Colab runtime, and downloads the public HBNN core implementation at a tested revision.


In [ ]:
%pip -q install geoopt==0.5.1

# Download the public BMLR implementation at the tested revision.
!test -d /content/mlss_hbnn || git clone -q https://github.com/GitZH-Chen/HBNN.git /content/mlss_hbnn
!git -C /content/mlss_hbnn checkout -q d5c79c8eed36a0b7c2f15e9a8fbcd0318216e5b0


In [ ]:
import sys
import warnings

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

warnings.filterwarnings("ignore", category=SyntaxWarning)
sys.path.insert(0, "/content/mlss_hbnn")
from lib.bnn.BMLR import BMLR
from lib.geoopt.manifolds.stereographic import PoincareBall
from lib.models.resnet import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Load MNIST

We resize each grayscale image to $32\times32$ and use 12,000 training examples for a short tutorial run. The test accuracy is still computed on the complete 10,000-image MNIST test set.


In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_data = datasets.MNIST("/content/data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("/content/data", train=False, download=True, transform=transform)
train_loader = DataLoader(Subset(train_data, range(12_000)), batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256)


## Define ResNet-18 + Poincaré BMLR

The only geometric hyperparameter exposed here is the clipping radius $r$.


In [ ]:
class ResNet18PoincareBMLR(nn.Module):
    def __init__(self, clip_radius=1.0, K=-1.0):
        super().__init__()
        self.encoder = resnet18(img_dim=[1, 32, 32], embed_dim=512, remove_linear=True)
        self.ball = PoincareBall(c=-K, learnable=False)
        self.head = BMLR(n_classes=10, dim=512, K=K, metric="poincare")
        self.clip_radius = clip_radius

    def forward(self, images):
        z = self.encoder(images)
        factor = torch.clamp(self.clip_radius / z.norm(dim=-1, keepdim=True), max=1.0)
        return self.head(self.ball.expmap0(factor * z))


model = ResNet18PoincareBMLR(clip_radius=1.0, K=-1.0).to(device)


## Train and test

This short teaching run is not the full paper protocol.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 3):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}: loss = {loss.item():.4f}")


In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        predictions = model(images.to(device)).argmax(dim=1).cpu()
        correct += (predictions == labels).sum().item()

print(f"Test accuracy: {correct / len(test_loader.dataset):.2%}")


## Modify and interpret

Change `clip_radius=1.0` to `0.5` or `2.0`, recreate the model, and rerun training.

- A smaller $r$ clips more Euclidean features and keeps their Poincaré images closer to the origin.
- A larger $r$ allows features to move closer to the boundary, where the exponential map can produce smaller back-propagated gradients.

**Interpret.** The image encoder remains Euclidean. Geometry enters only at the interface `CLIP → Exp₀ → BMLR`, so this notebook isolates the effect of a hyperbolic classification head.


## Takeaways

- ResNet-18 maps MNIST images to 512-dimensional Euclidean features.
- Clipping controls the norm of each feature before the exponential map.
- The exponential map sends clipped features to the Poincaré ball.
- BMLR converts Poincaré features into ten class logits through Busemann functions.
- The complete hybrid model is trained end to end with ordinary cross-entropy loss.
